* [gps visualizer: Smoothing & Simplifying Tracks](https://www.gpsvisualizer.com/tutorials/track_filters.php)
* [gps visualizer: Calculating Elevation Gain](https://www.gpsvisualizer.com/tutorials/elevation_gain.php)
* [x-plane flask/websockets](https://austinsnerdythings.com/2024/03/24/adding-some-polish-to-the-x-plane-python-autopilot-with-flask-redis-and-websockets/)
* [x-plane cross track error](https://austinsnerdythings.com/2024/04/06/adding-track-following-direct-to-with-cross-track-error-to-the-python-x-plane-autopilot/)
* [Calculate distance, bearing and more between Latitude/Longitude points](https://www.movable-type.co.uk/scripts/latlong.html)
* [Ramer-Douglas-Peucker - line simplification algorithm](https://en.wikipedia.org/wiki/Ramer–Douglas–Peucker_algorithm)

In [34]:
from typing import Any, Dict
import json
import sys
import math
import os
os.environ["CALORIES_DEBUG"] = 'true'
print(os.environ.get("CALORIES_DEBUG", "nope"))

true


In [35]:
from pathlib import Path
current_dir = Path().resolve()
src_dir = current_dir.parent
root_dir = parent_dir.parent
data_dir = str(src_dir) + '/' + 'data'
print(f'root_dir: {root_dir}')
print(f'data_dir: {data_dir}')
print(f'parent_dir: {src_dir}')
print(f'current_dir: {current_dir}')
sys.path.append(str(root_dir))
print(sys.path)
from ramer_douglas_peucker import ramer_douglas_peucker
# from calories import *
# from calories import pandolfCalories, lcdaCalories, minimumMechanicsCalories, calorieEnsemble, load_sample_data, models

root_dir: /Users/matt/@mattduffy/calories-py
data_dir: /Users/matt/@mattduffy/calories-py/src/data
parent_dir: /Users/matt/@mattduffy/calories-py/src
current_dir: /Users/matt/@mattduffy/calories-py/src/tests
['/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14', '/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload', '', '/Users/matt/@mattduffy/calories-py/lib/python3.14/site-packages', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py', '/Users/matt/@mattduffy/calories-py']


In [43]:
with open('../data/walk_31-i-m-on-unifi.json', 'r') as file:
    contents = file.read()
    walk_31 = json.loads(contents)

coords = walk_31["features"][0]["geometry"]["coordinates"]
print(coords[1])

[-122.20907615758304, 37.82453156141039, 87.28833691863991, 178.11343436688185, 22.783305624032607, 1784926645013]


In [38]:
def save_simplified_file(data: Dict[str, Any], name: str) -> None:
    path = data_dir + "/" + name
    print(path)
    with open(path, 'x') as f:
        f.write(json.dumps(data))

# save_simplified_file({"path": "shorter"}, "walk_31-simplified.json")

In [7]:
def degs2Rads(degs: float) -> float:
    return degs * math.pi/180

def rads2Degs(rads: float) -> float:
    return rads * 180 / math.pi

rads2Degs(degs2Rads(-180))

-180.0

In [39]:
def _cross_track_distance(point: list[float], track: list[list[float]], scale='m') -> float:
    """Calculate the perpendicular distance (in meters) of a gps point from a track (polyline 
        segment) consisting of two other gps points.

    Args:
        point (list[float]): A list containing gps coordinates [lon, lat].
        track (list[list[float]]): A two point polyline list [[lon, lat], [lon, lat]].
        scale (str): Sets the unit size of the track distance value, either 'km' or 'm'.

    Returns:
        float: The perpendicular distance of the point from the track (line segment).    
    """
    # Convert all longitudes and latitudes from degrees to radians
    point_lon, point_lat = math.radians(point[0]), math.radians(point[1])
    start_lon, start_lat = math.radians(track[0][0]), math.radians(track[0][1])
    end_lon, end_lat = math.radians(track[1][0]), math.radians(track[1][1])

    # Calculate the angular distance from start to point
    # Ensure the argument is within the domain of acos
    acos_argument = math.sin(start_lat) * math.sin(point_lat) + math.cos(start_lat) * math.cos(point_lat) * math.cos(point_lon - start_lon)
    acos_argument = max(-1, min(1, acos_argument))  # Clamp the argument between -1 and 1
    delta_sigma = math.acos(acos_argument)

    # Calculate the bearing from start to point and start to end
    theta_point = math.atan2(math.sin(point_lon - start_lon) * math.cos(point_lat),
                             math.cos(start_lat) * math.sin(point_lat) - math.sin(start_lat) * math.cos(point_lat) * math.cos(point_lon - start_lon))
    theta_end = math.atan2(math.sin(end_lon - start_lon) * math.cos(end_lat),
                           math.cos(start_lat) * math.sin(end_lat) - math.sin(start_lat) * math.cos(end_lat) * math.cos(end_lon - start_lon))

    # Calculate the cross track distance
    cross_track_dist = math.asin(math.sin(delta_sigma) * math.sin(theta_point - theta_end))
    earth_radius_in_km = 6371
    scale = 1 if (scale == 'km') else 1000
    # print(f'scale = {scale}')
    cross_track_dist = cross_track_dist * (earth_radius_in_km * scale)
    return cross_track_dist


In [7]:
kbjc_runways = {
    "30R/12L": {
        "Runway 12L": {
            "Latitude": 39.91529286666667,
            "Longitude": -105.12841313333334
        },
        "Runway 30R": {
            "Latitude": 39.901373883333335,
            "Longitude": -105.10191808333333
        }
    }
}
test_locations = {
	"kbjc_tower": {
        "lat": 39.9064,
		"lon": -105.1208
	},  
}
kbjc_runway_30R_start = (kbjc_runways["30R/12L"]["Runway 30R"]["Longitude"], kbjc_runways["30R/12L"]["Runway 30R"]["Latitude"])
kbjc_runway_30R_end = (kbjc_runways["30R/12L"]["Runway 12L"]["Longitude"], kbjc_runways["30R/12L"]["Runway 12L"]["Latitude"])
kbjc_tower = (test_locations["kbjc_tower"]["lon"], test_locations["kbjc_tower"]["lat"])
def test_cross_track_distance():
    print(f"start lon: {kbjc_runway_30R_start[0]}, start lat: {kbjc_runway_30R_start[1]}")
    print(f"end lon: {kbjc_runway_30R_end[0]}, end lat: {kbjc_runway_30R_end[1]}")
    print(f"tower lon: {kbjc_tower[0]}, tower lat: {kbjc_tower[1]}")
    dist = _cross_track_distance(kbjc_tower, (kbjc_runway_30R_start, kbjc_runway_30R_end))
    print(f"cross track distance: {dist}")
test_cross_track_distance()

p1 = (-122.20130, 37.82531)
p2 = (-122.20263, 37.82850)
seg = list((coords[10], coords[int(len(coords)/2)]))
print(f'p1 {cross_track_distance(p1, seg)}')
print(f'p2 {int(cross_track_distance(p2, seg))}')

start lon: -105.10191808333333, start lat: 39.901373883333335
end lon: -105.12841313333334, end lat: 39.91529286666667
tower lon: -105.1208, tower lat: 39.9064
cross track distance: -449.05534866747985


NameError: name 'cross_track_distance' is not defined

In [ ]:
def _ramer_douglas_peucker(points: list[list[float]] = (), epsilon: float = 2.0) -> list[list[float]]:
    """A cartographic generalization algorithm to reduce the number of points in a 
       curve, composed of line segments, into a similar curve consisting of fewer points.

    Args:
        points (list[list[float]]): A list of gps coordinate arrays.
        epsilon (float): A distance threshold value > 0.

    Returns:
        list[list[float]]: A new list of gps points, of equal length as points, or fewer.
    """
    max_distance = 0
    index = 0
    end = len(points)
    #print(f'len(points): {len(points)}, epsilon: {epsilon}, max_distance: {max_distance}, index: {index}, end: {end}')
    for i in range(1, end - 1):
        #print(f'({i}) lon {points[i][0]}, lat {points[i][1]}')
        #print(f'point[i: {i}], (points[0], points[end: {end - 1}]')
        d = abs(_cross_track_distance(points[i], (points[0], points[end - 1])))
        #print(f'point[{i}] has perpendicular distance {d} from points polyline.')
        if (d > max_distance):
            index = i
            max_distance = d
            #print(f'\tnew max_distance {max_distance}, at index {index}')
        #print('\n')
        
    if (max_distance > epsilon):
        # print(f'max_distance {max_distance} > epsilon {epsilon}')
        # print(f'making recursive calls')
        # recursive call to ramer_douglas_peucker()
        # left half spans points[0]..points[index]; right half spans points[index]..points[end - 1].
        # The pivot point[index] is shared by both halves.
        recResultsLeft = list(ramer_douglas_peucker(points[0:index + 1], epsilon))
        recResultsRight = list(ramer_douglas_peucker(points[index:end], epsilon))

        # combine the two results lists, dropping the shared pivot (last of left == first of right)
        # print('recursively combined simplified_list')
        #print(f'left {len(recResultsLeft)}', recResultsLeft, recResultsLeft[:-1])
        #print(f'right {len(recResultsRight)}', recResultsRight)
        simplified_list = recResultsLeft[:-1] + recResultsRight
        # print(f'combined, {simplified_list}')
    else:
        # print(f'max_distance {max_distance} !> epsilon {epsilon}')
        simplified_list = list((points[0], points[end - 1]))
        
    return simplified_list


In [44]:
full = len(coords)
start = 0
end = full
line = coords[start:end]
print(len(line))
simplified = ramer_douglas_peucker(line, 0.9)
print(f'original list: {len(line)}, new list: {len(simplified)}')

7474
original list: 7474, new list: 1258


In [57]:
data = dict(walk_31)
data["features"][0]["geometry"]["coordinates"] = simplified
type(data)
save_simplified_file(data, 'walk_31_simplified.json')

/Users/matt/@mattduffy/calories-py/src/data/walk_31_simplified.json
